# **RQ1**: Number of neighbors per node over time

For each snapshot, I compute the node degrees and generate a snapshot-specific file .csv that maps each node (word label) to its degree in that snapshot

In [ ]:
import pandas as pd
import networkx as nx
import time
from pathlib import Path


edges_dir = Path(r"../edges_analysis/edges")
vec_dir = Path(r"../embedding/word_embeddings_cleaned")
output_dir = Path(r"degree_csv")
output_dir.mkdir(exist_ok=True)

percentile = 99  
t0_total = time.time()
print(f"[START] Elaborazione batch snapshot 1-10 (percentile {percentile})")

for snap in range(1, 11):
    t0 = time.time()
    
    edges_tsv = edges_dir / f"edges_snap{snap}_p{percentile}.tsv"
    vec_file = vec_dir / f"fasttext_snap{snap}_filt.vec"
    output_csv = output_dir / f"node_degree_snap{snap}_p{percentile}.csv"
    
    print(f"\n[SNAPSHOT {snap}] Lettura file archi: {edges_tsv.name}")
    
    # archi
    df_edges = pd.read_csv(edges_tsv, sep='\t')
    print(f"[INFO] Righe archi lette: {len(df_edges)}")
    
    # token
    tokens = []
    with open(vec_file, 'r', encoding='utf-8') as f:
        next(f)  # salta la prima riga
        for line in f:
            token = line.strip().split()[0]
            tokens.append(token)
    print(f"[INFO] Token letti: {len(tokens)}")
    
    # Determina colonne source, target, weight 
    col_source = df_edges.columns[0]
    col_target = df_edges.columns[1]
    col_weight = df_edges.columns[2] if len(df_edges.columns) > 2 else None
    
    # Crea grafo non diretto 
    if col_weight:
        G = nx.from_pandas_edgelist(
            df_edges,
            source=col_source,
            target=col_target,
            edge_attr=col_weight,
            create_using=nx.Graph()
        )
    else:
        G = nx.from_pandas_edgelist(
            df_edges,
            source=col_source,
            target=col_target,
            create_using=nx.Graph()
        )
    print(f"[INFO] Grafo creato - Nodi: {G.number_of_nodes()}, Archi: {G.number_of_edges()}")
    
    # Calcola degree e aggiungi token 
    df_degree = pd.DataFrame(G.degree(), columns=["node_id","degree"])
    df_degree["token"] = df_degree["node_id"].apply(lambda x: tokens[x])
    
    # --- 6. Salva CSV ---
    df_degree.to_csv(output_csv, index=False)
    print(f"[DONE] CSV salvato: {output_csv.name} in {time.time() - t0:.2f}s")

print(f"\n[TOTAL] Tempo totale batch: {time.time() - t0_total:.2f}s")


We compute the degree of each node for every snapshot and retain only the words that appear in all ten snapshots. 
For these common words, we construct a word–snapshot matrix in which each row corresponds to a word and each column 
reports its degree in a given snapshot. This representation enables a consistent longitudinal analysis of node 
connectivity over time, independent of snapshot-specific node indexing.


In [1]:
import pandas as pd
from pathlib import Path


degree_dir = Path("degree_csv")
n_snapshots = 10

print("[START] Costruzione matrice token × snapshot (degree)")

# Carica tutti i CSV in un dizionario 
dfs = {}

for snap in range(1, n_snapshots + 1):
    csv_path = degree_dir / f"node_degree_snap{snap}_p99.csv"
    df = pd.read_csv(csv_path)
    
    # teniamo solo token e degree
    df = df[["token", "degree"]].rename(
        columns={"degree": f"degree_snap{snap}"}
    )
    
    dfs[snap] = df
    print(f"[INFO] Snapshot {snap} caricato: {len(df)} token")

# Intersezione dei token 
df_common = dfs[1]

for snap in range(2, n_snapshots + 1):
    df_common = df_common.merge(
        dfs[snap],
        on="token",
        how="inner"
    )

print(f"[DONE] Token comuni a tutti gli snapshot: {len(df_common)}")

# Riordina colonne 
degree_cols = [f"degree_snap{snap}" for snap in range(1, n_snapshots + 1)]
df_common = df_common[["token"] + degree_cols]


output_path = degree_dir / "token_degree_all_snapshots.csv"
df_common.to_csv(output_path, index=False)

print(f"[SAVED] File finale salvato: {output_path.name}")


[START] Costruzione matrice token × snapshot (degree)
[INFO] Snapshot 1 caricato: 49460 token
[INFO] Snapshot 2 caricato: 47189 token
[INFO] Snapshot 3 caricato: 64187 token
[INFO] Snapshot 4 caricato: 64631 token
[INFO] Snapshot 5 caricato: 75488 token
[INFO] Snapshot 6 caricato: 82826 token
[INFO] Snapshot 7 caricato: 64187 token
[INFO] Snapshot 8 caricato: 58410 token
[INFO] Snapshot 9 caricato: 52701 token
[INFO] Snapshot 10 caricato: 48872 token
[DONE] Token comuni a tutti gli snapshot: 29785
[SAVED] File finale salvato: token_degree_all_snapshots.csv
